In [ ]:
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input, GRU, Conv1D, MaxPool1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score, mean_squared_error, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score
from tensorflow.keras.losses import Huber
import pandas as pd
import numpy as np
import pickle

In [5]:
df = pd.read_csv("NIFTY.csv")
print(df.shape)
df.head()

(4977, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,NaN,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100,NaN,NaN,NaN,NaN
1,2005-11-30,2605.10009765625,2605.110107421875,2605.10009765625,2605.110107421875,0,^CNX100,NaN,NaN,NaN
2,2005-12-01,2649.75,2654.10009765625,2595.449951171875,2614.64990234375,0,^CNX100,1.713942,NaN,NaN
3,2005-12-02,2652.300048828125,2679.89990234375,2646.89990234375,2670.35009765625,0,^CNX100,0.096237,NaN,NaN
4,2005-12-05,2619.699951171875,2663.550048828125,2614.050048828125,2663.550048828125,0,^CNX100,-1.229126,NaN,NaN


In [6]:
df = df.iloc[1:4977,:]
print(df.shape)
df.head()

(4976, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
1,2005-11-30,2605.10009765625,2605.110107421875,2605.10009765625,2605.110107421875,0,^CNX100,NaN,NaN,NaN
2,2005-12-01,2649.75,2654.10009765625,2595.449951171875,2614.64990234375,0,^CNX100,1.713942,NaN,NaN
3,2005-12-02,2652.300048828125,2679.89990234375,2646.89990234375,2670.35009765625,0,^CNX100,0.096237,NaN,NaN
4,2005-12-05,2619.699951171875,2663.550048828125,2614.050048828125,2663.550048828125,0,^CNX100,-1.229126,NaN,NaN
5,2005-12-06,2618.60009765625,2646.449951171875,2604.050048828125,2622.300048828125,0,^CNX100,-0.041984,NaN,NaN


In [7]:
df.isna().sum()

Date                0
Close               0
High                0
Low                 0
Open                0
Volume              0
Ticker              0
Daily_Return_%      1
MA_50              49
MA_200            199
dtype: int64

In [8]:
colist = ['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%']
for col in colist:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"])
for col in colist:
    print(df[col].dtype)

float64
float64
float64
float64
int64
float64


#Feature Engineering

In [9]:
for col in df.columns:
    print(col, "->",df[col].dtype,)

Date -> datetime64[us]
Close -> float64
High -> float64
Low -> float64
Open -> float64
Volume -> int64
Ticker -> str
Daily_Return_% -> float64
MA_50 -> float64
MA_200 -> float64


In [10]:
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
0,2005-11-30,2605.100098,2605.110107,2605.100098,2605.110107,0,^CNX100,NaN,NaN,NaN
1,2005-12-01,2649.750000,2654.100098,2595.449951,2614.649902,0,^CNX100,1.713942,NaN,NaN
2,2005-12-02,2652.300049,2679.899902,2646.899902,2670.350098,0,^CNX100,0.096237,NaN,NaN
3,2005-12-05,2619.699951,2663.550049,2614.050049,2663.550049,0,^CNX100,-1.229126,NaN,NaN
4,2005-12-06,2618.600098,2646.449951,2604.050049,2622.300049,0,^CNX100,-0.041984,NaN,NaN


In [11]:
#Basic Columns
df["NIFTY_100"] = df["Close"].pct_change()
df["Volatility_100"] = (df["NIFTY_100"].rolling(20).std())

prev_close = df["Close"].shift(-1)
df["TR"] = pd.concat(
    [
        df["High"] - df["Close"], 
        (df["High"] - prev_close).abs(), 
        (df["Low"] - prev_close).abs()
    ], axis=1
).max(axis=1)

#Rolling volume
df["Rolling Volume"] = df.groupby("Ticker")["Volume"].transform(lambda x : x.rolling(20).mean())

#Rolling return
df["Rolling_Return_5D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(5))
df["Rolling_Return_10D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(10))
df["Rolling_Return_20D"] = df.groupby("Ticker")["Close"].transform(lambda x: x.pct_change(20))

#Ranges
df["High_low_%"] = ((df["High"] - df["Low"])/df["Close"])*100
df["Open_close_%"] = ((df["Close"] - df["Open"]).abs()/df["Close"])*100

#Lagged Volatility
df["Rolling_Volatility_20D"] = df.groupby("Ticker")["Daily_Return_%"].transform(lambda x: x.rolling(20).std())
df["Lagged_Volatility_1"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(1))
df["Lagged_Volatility_5"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(5))
df["Lagged_Volatility_10"] = df.groupby("Ticker")["Rolling_Volatility_20D"].transform(lambda x: x.shift(10))

#MA_to_Price
df["MA50_to_Price"] = (df["MA_50"]/df["Close"])
df["MA200_to_Price"] = (df["MA_200"]/df["Close"])

#Volatility
df["Daily_Return"] = df.groupby("Ticker")["Close"].pct_change()

df["Rolling_Volatility_5D"] = df.groupby("Ticker")["Daily_Return"].transform(lambda x: x.rolling(5).std())
df["Rolling_Volatility_10D"] = df.groupby("Ticker")["Daily_Return"].transform(lambda x: x.rolling(10).std())

def future_shift(col):
    return col.iloc[::-1].rolling(5).std().iloc[::-1].shift(-1)

#Final prediction that is to be made
df["future_shift_5"] = df.groupby("Ticker")["Daily_Return"].transform(future_shift)

In [12]:
print(df[["Date", "Daily_Return_%", "future_shift_5"]].tail(10))

           Date  Daily_Return_%  future_shift_5
4966 2026-02-09        0.724409        0.008416
4967 2026-02-10        0.242749        0.008399
4968 2026-02-11        0.151405        0.008698
4969 2026-02-12       -0.542450        0.010744
4970 2026-02-13       -1.344713        0.009155
4971 2026-02-16        0.854460             NaN
4972 2026-02-17        0.227054             NaN
4973 2026-02-18        0.415164             NaN
4974 2026-02-19       -1.487281             NaN
4975 2026-02-20        0.485236             NaN


In [13]:
df = df.dropna()
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200,...,Rolling_Volatility_20D,Lagged_Volatility_1,Lagged_Volatility_5,Lagged_Volatility_10,MA50_to_Price,MA200_to_Price,Daily_Return,Rolling_Volatility_5D,Rolling_Volatility_10D,future_shift_5
199,2006-09-19,3348.350098,3403.899902,3331.300049,3394.800049,0,^CNX100,-1.047637,3153.158994,3058.710750,...,1.065387,1.028316,0.980829,0.697964,0.941705,0.913498,-0.010476,0.010730,0.013598,0.009519
200,2006-09-20,3388.800049,3394.350098,3314.100098,3318.750000,0,^CNX100,1.208056,3161.117993,3062.629249,...,1.053750,1.065387,1.028683,0.623630,0.932813,0.903750,0.012081,0.008237,0.014144,0.008916
201,2006-09-21,3436.300049,3439.800049,3413.500000,3413.550049,0,^CNX100,1.401676,3168.625996,3066.562000,...,1.072433,1.053750,1.029501,0.625134,0.922104,0.892402,0.014017,0.009742,0.014483,0.007153
202,2006-09-22,3427.949951,3445.550049,3410.750000,3414.649902,0,^CNX100,-0.242997,3176.421997,3070.440249,...,1.076358,1.072433,1.029120,0.626083,0.926624,0.895707,-0.002430,0.010185,0.014550,0.006808
203,2006-09-25,3407.300049,3435.649902,3400.050049,3435.250000,0,^CNX100,-0.602398,3184.655000,3074.378250,...,1.091045,1.076358,1.028316,0.999139,0.934656,0.902292,-0.006024,0.011001,0.009316,0.006344


#Training and Testing

In [28]:
drop_cols = ["NIFTY_100", "Rolling_Volatility_20D"]
X = df.drop(["future_shift_5", "Ticker", "Date", "Daily_Return"]+drop_cols, axis = 1)
Y = df["future_shift_5"]

mxscale = MinMaxScaler()
X_scaled = mxscale.fit_transform(X)

Xtrain, Xtest, Ytrain, Ytest = train_test_split(X_scaled, Y, test_size=0.2, random_state = 42)

print(Xtrain.shape)
print(Xtest.shape)
print(Ytrain.shape)
print(Ytest.shape)

(3817, 23)
(955, 23)
(3817,)
(955,)


In [29]:
lightgbm = LGBMRegressor(
    n_estimators=1000, max_depth=20, random_state=42, min_child_samples=10
)

tscv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

score = cross_val_score(
    lightgbm,
    Xtrain, 
    Ytrain,
    cv=tscv,
    scoring="r2",
    n_jobs=-1
)

lightgbm.fit(Xtrain, Ytrain)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000710 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5865
[LightGBM] [Info] Number of data points in the train set: 3817, number of used features: 23
[LightGBM] [Info] Start training from score 0.010300


,max_depth,20
,n_estimators,1000
,min_child_samples,10
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,learning_rate,0.1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [30]:
# params = {
#     "n_estimators": [100, 500, 1000],
#     "min_child_samples": [10, 20, 25],
#     "min_s": [5, 10],
#     "min_samples_split": [1, 2, 5],
#     "max_features": [1.0, "sqrt", "log"],
#     "bootstrap": [True, False],
#     "criterion": ["squared_error", "absolute_error"],
#     "max_samples": [None, 0.7, 0.8]
# }

# Lparams = {
#     "n_estimators": [500, 1000],
#     "max_depth": [5, 10],
#     "num_leaves": [10, 20],
#     "min_child_samples": [10, 20],
#     "learning_rate": [0.05, 0.1]
# }

# gmodel = GridSearchCV(lightgbm, param_grid=Lparams, n_jobs=-1, cv=3)
# gmodel.fit(Xtrain, Ytrain)
# print(gmodel.best_params_)

In [31]:
# Ypred = gmodel.predict(Xtest)
# print("mse", mean_squared_error(Ytest, Ypred))
# print("mae", mean_absolute_error(Ytest, Ypred))
# print("r2score", r2_score(Ytest, Ypred)*100)

Ypred = lightgbm.predict(Xtest)
print("mse",mean_squared_error(Ytest, Ypred))
print("mae",mean_absolute_error(Ytest, Ypred))
print("r2score",r2_score(Ytest, Ypred)*100)

print(score)
print(score.mean())

mse 1.2468159496994991e-05
mae 0.002401285509396495
r2score 72.78983684174722
[0.79502625 0.74767113 0.74859761 0.7038609  0.72768481]
0.7445681412718297


In [32]:
X.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%', 'MA_50',
       'MA_200', 'Volatility_100', 'TR', 'Rolling Volume', 'Rolling_Return_5D',
       'Rolling_Return_10D', 'Rolling_Return_20D', 'High_low_%',
       'Open_close_%', 'Lagged_Volatility_1', 'Lagged_Volatility_5',
       'Lagged_Volatility_10', 'MA50_to_Price', 'MA200_to_Price',
       'Rolling_Volatility_5D', 'Rolling_Volatility_10D'],
      dtype='str')

#Improving the model

In [33]:
importance = pd.Series(
    lightgbm.feature_importances_,
    index = X.columns
).sort_values(ascending=False)

print(importance)

TR                        2007
Daily_Return_%            1832
Rolling_Volatility_5D     1800
Rolling_Return_5D         1784
Rolling_Return_10D        1775
Open_close_%              1718
High_low_%                1681
Rolling_Return_20D        1658
MA200_to_Price            1652
MA50_to_Price             1638
Rolling_Volatility_10D    1528
Volume                    1515
Rolling Volume            1476
Lagged_Volatility_10      1444
Lagged_Volatility_5       1263
Volatility_100            1083
Lagged_Volatility_1       1042
MA_200                     748
MA_50                      732
Close                      673
High                       333
Open                       310
Low                        308
dtype: int32


#Using LSTM

In [37]:
dfcopy = df
cols = ['Close', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return_%', 'MA_50', 'MA_200', 'Volatility_100',
       'TR', 'Rolling Volume', 'Rolling_Return_5D', 'Rolling_Return_10D',
       'Rolling_Return_20D', 'High_low_%', 'Open_close_%',
       'Lagged_Volatility_1', 'Lagged_Volatility_5', 'Lagged_Volatility_10',
       'MA50_to_Price', 'MA200_to_Price', 'Rolling_Volatility_5D', 'Rolling_Volatility_10D']

In [38]:
dfcopy.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200,...,Rolling_Volatility_20D,Lagged_Volatility_1,Lagged_Volatility_5,Lagged_Volatility_10,MA50_to_Price,MA200_to_Price,Daily_Return,Rolling_Volatility_5D,Rolling_Volatility_10D,future_shift_5
199,2006-09-19,3348.350098,3403.899902,3331.300049,3394.800049,0,^CNX100,-1.047637,3153.158994,3058.710750,...,1.065387,1.028316,0.980829,0.697964,0.941705,0.913498,-0.010476,0.010730,0.013598,0.009519
200,2006-09-20,3388.800049,3394.350098,3314.100098,3318.750000,0,^CNX100,1.208056,3161.117993,3062.629249,...,1.053750,1.065387,1.028683,0.623630,0.932813,0.903750,0.012081,0.008237,0.014144,0.008916
201,2006-09-21,3436.300049,3439.800049,3413.500000,3413.550049,0,^CNX100,1.401676,3168.625996,3066.562000,...,1.072433,1.053750,1.029501,0.625134,0.922104,0.892402,0.014017,0.009742,0.014483,0.007153
202,2006-09-22,3427.949951,3445.550049,3410.750000,3414.649902,0,^CNX100,-0.242997,3176.421997,3070.440249,...,1.076358,1.072433,1.029120,0.626083,0.926624,0.895707,-0.002430,0.010185,0.014550,0.006808
203,2006-09-25,3407.300049,3435.649902,3400.050049,3435.250000,0,^CNX100,-0.602398,3184.655000,3074.378250,...,1.091045,1.076358,1.028316,0.999139,0.934656,0.902292,-0.006024,0.011001,0.009316,0.006344


In [39]:
for ticker, data in dfcopy.groupby("Ticker"):
    Xsc = []
    Ysc = []

    X = data[cols].values
    Y = data["future_shift_5"]
    msc = MinMaxScaler()
    Xmsc = msc.fit_transform(X)

    window = 60
    for i in range(window, len(X)):
        Xsc.append(Xmsc[i-window:i])
        Ysc.append(Y.iloc[i])

    Xsc = np.array(Xsc)
    Ysc = np.array(Ysc)

In [40]:
Xsc.dtype

dtype('float64')

In [41]:
split = int(len(Xsc)*0.8)

XLTrain = Xsc[:split]
XLTest = Xsc[split:]

YLTrain = Ysc[:split]
YLTest = Ysc[split:]

In [42]:
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(
    YLTrain.reshape(-1,1)
)

yt_scaled = y_scaler.transform(
    YLTest.reshape(-1,1)
)

In [43]:
XLTrain.dtype
YLTrain.dtype

dtype('float64')

In [44]:
print("XLTrain", XLTrain.shape)
print("XLTest", XLTest.shape)
print("YLTrain", YLTrain.shape)
print("YLTest", YLTest.shape)

XLTrain (3769, 60, 23)
XLTest (943, 60, 23)
YLTrain (3769,)
YLTest (943,)


In [ ]:
modelL = Sequential(
    [Input(shape=(60, XLTrain.shape[2])),
    GRU(64), 
    # GRU(64, return_sequences=True)
    Dropout(0.2),
    # GRU(64),
    # Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)]
)

modelL.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss=Huber()
)

call_back = [EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)
    # ReduceLROnPlateau(
    #     moniter='val_loss',
    #     patience=5,
    #     factor=0.5,
    #     min_LR=1e-6
    # )
]
modelL.fit(
    XLTrain, 
    y_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=call_back
)

# modelC = Sequential([
#     Input(shape=(60, XLTrain.shape[2])),
#     Conv1D(filters=64, kernel_size=5, activation='relu'),
#     MaxPool1D(pool_size=2),
#     Dropout(0.2),

#     Conv1D(filters=128, kernel_size=5, activation='relu'),
#     MaxPool1D(pool_size=2),
#     Dropout(0.2),

#     Flatten(),

#     Dense(64, activation='relu'),
#     Dropout(0.2),

#     Dense(32, activation='relu'),
#     Dense(1)]
# )

# modelC.compile(
#     optimizer=Adam(learning_rate=0.0005),
#     loss="mse",
#     metrics=["mse"]
# )

# modelC.fit(
#     XLTrain,
#     YLTrain,
#     epochs=50,
#     batch_size=32,
#     validation_split = 0.1,
#     callbacks = call_back
# )

Epoch 1/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 9s 53ms/step - loss: 0.0078 - val_loss: 0.0052
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.0065 - val_loss: 0.0041
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 0.0063 - val_loss: 0.0040
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - loss: 0.0060 - val_loss: 0.0043
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - loss: 0.0061 - val_loss: 0.0045
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 0.0057 - val_loss: 0.0045
Epoch 7/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0058 - val_loss: 0.0051
Epoch 8/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - loss: 0.0058 - val_loss: 0.0048
Epoch 9/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.0058 - val_loss: 0.0043
Epoch 10/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0057 - val_loss: 0.0050
Epoch 11/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.0056 - val_loss: 0.0041


In [55]:
Ypred = modelL.predict(XLTest)
pred = y_scaler.inverse_transform(Ypred)
print("r2score",r2_score(YLTest, pred))
print("mae",mean_absolute_error(YLTest, pred))

print("Prediction mean and std", pred.mean(), pred.std())
print("Test mean and std", YLTest.mean(), YLTest.std())

30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
r2score -2.3123335990837464
mae 0.006759730667034993
Prediction mean and std 0.0129158795 0.0028184045
Test mean and std 0.007059997129365867 0.004262675014094219


In [34]:
Ypred = modelC.predict(XLTest)
# pred = y_scaler.inverse_transform(Ypred)
print("r2score",r2_score(YLTest, Ypred))
print("mae",mean_absolute_error(YLTest, Ypred))

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
r2score -2.0778620357610214
mae 0.005758641748989651


In [35]:
print("Actual:", YLTest[:10])
print("Predicted:", Ypred[:10].flatten())

Actual: [0.01410684 0.01382924 0.01452269 0.01494132 0.01443347 0.01397927
 0.01216147 0.01322085 0.01336921 0.01020335]
Predicted: [0.00902836 0.00872875 0.00896194 0.00920199 0.0103381  0.01138048
 0.01209394 0.01196217 0.01168162 0.01097497]


In [36]:
print(XLTrain.shape, XLTest.shape)
print(YLTrain.shape, YLTest.shape)
print("Actual:", YLTest[:10])
print("Predicted:", Ypred[:10].flatten())

(3769, 60, 23) (943, 60, 23)
(3769,) (943,)
Actual: [0.01410684 0.01382924 0.01452269 0.01494132 0.01443347 0.01397927
 0.01216147 0.01322085 0.01336921 0.01020335]
Predicted: [0.00902836 0.00872875 0.00896194 0.00920199 0.0103381  0.01138048
 0.01209394 0.01196217 0.01168162 0.01097497]


In [37]:
print("Actual mean:", YLTest.mean())
print("Actual std:", YLTest.std())

print("Pred mean:", pred.mean())
print("Pred std:", pred.std())

print("Actual min/max:", YLTest.min(), YLTest.max())
print("Pred min/max:", pred.min(), pred.max())

Actual mean: 0.007059997129365867
Actual std: 0.004262675014094219
Pred mean: 0.012804042
Pred std: 0.0028643003
Actual min/max: 0.0009394142581399712 0.04211738838320969
Pred min/max: 0.008954329 0.037147686


In [38]:
baseline = np.full_like(YLTest, YLTrain.mean())

print("Baseline MAE:",
      mean_absolute_error(YLTest, baseline))

print("Baseline R2:",
      r2_score(YLTest, baseline))

Baseline MAE: 0.004993804711517428
Baseline R2: -0.8513850182717915
